# Models

In [67]:
import numpy as np
from scipy.stats import gamma, norm
# get truncated gamma and normal distributions
from scipy.stats import truncnorm
# from truncnorm get gamma and normal distributions


class SlopeMAPPerceptual:
    def __init__(self, *args, **kwargs):
        # Leak parameter
        self.leak = kwargs.get('leak', 0)
        # x values = forcefield values
        self.x = kwargs.get('x', np.linspace(-1, 1, 9).round(2))
        if not isinstance(self.x, np.ndarray):
            self.x = np.array(list(self.x))
        
        # Initialize outcome-based prior
        # Slope prior mean and standard deviation
        self.slope_prior_mean = kwargs.get('slope_prior_mean', 0.0)  # Default mean to 0
        self.slope_prior_std = kwargs.get('slope_prior_std', 1.0)    # Default std to 1.0
        
        # Ensure the prior standard deviation is positive
        self.slope_prior_std = max(0.001, self.slope_prior_std)  # Set a minimum value
        
        # Possible values for the slope
        self.slope_range = kwargs.get('slope_range', np.arange(-10, 10, 0.1))
        
        # Initialize log-posterior for the slope with prior
        self.lp_slope = np.log(norm.pdf(self.slope_range, self.slope_prior_mean, self.slope_prior_std))
        
        # Define logit function
        self.logit = lambda x: 1 / (1 + np.exp(x))

        self.binary = kwargs.get('binary', True)
        
    def perceptual_update(self, choice, outcome, spaceship_estimate=None):
        """Update the log-posterior of the slope based on observed data."""
        if self.binary:
            v = self.slope_range * choice
            v = v - v * 2 * outcome  # Switch to negative if forcefield destroyed
    
            # Binary outcome likelihood
            ll = np.log(self.logit(v))
        else:
           # Continuous outcome: outcome = sigmoid(slope * x) * spaceship reward
            logits = self.slope_range * choice

            # logistic probabilities
            p = self.logit(logits)

            # in the generative process of exp. 3, spaceship rewards is
            # multiplied by shield value to produce an outcome
            # Use observed outcome and estimated spaceship to derive implied sigmoid output
            pred_outcomes = np.clip(outcome / spaceship_estimate, 1e-6, 1 - 1e-6)
            # pred_outcomes = p * (spaceship_estimate * 2)

            # gaussian likelihood: compare each pred_outcome to the obtained outcome
            likelihoods = np.exp(-0.5 * ((p-pred_outcomes) ** 2) / 0.1)
            # likelihoods = np.exp(-0.5 * ((outcome - pred_outcomes) ** 2) / 0.2)

            # posterior
            posterior = likelihoods / np.sum(likelihoods)

            ll = np.log(posterior)

        # Update log posterior
        self.lp_slope += ll
           
    def predict_shield_p(self, shields):
        """Predict outcomes (linear or logit) for given forcefields."""
        # Predict for 2 displayed forcefields
        x = np.arange(len(self.x))
        to_select = x[np.isin(self.x, shields)]
        slope = -self.get_slope() * int(self.binary) + self.get_slope() * int(not self.binary)
        return self.logit(slope * self.x[to_select])
    
    def get_slope(self):
        """Compute the current slope as the weighted mean of the slope range."""
        w = np.exp(self.lp_slope - np.max(self.lp_slope))
        slope = np.sum(w * self.slope_range) / np.sum(w)
        return slope


class NormativeValue:
    def __init__(self, spaceship_ids, mu0=.5, kappa0=1, alpha0=2, beta0=0.1):
        """
        Bayesian reward model using Normal-Gamma conjugate prior for multiple spaceships.

        Parameters:
            spaceship_ids : list of spaceship identifiers (ints or strings)
            mu0, kappa0, alpha0, beta0 : prior hyperparameters for all spaceships
        """
        self.mu0 = mu0
        self.kappa0 = kappa0
        self.alpha0 = alpha0
        self.beta0 = beta0

        self.data = {sid: [] for sid in spaceship_ids}

    def value_update(self, spaceship_id, reward):
        """
        Add an observed reward for a specific spaceship.

        Parameters:
            spaceship_id : ID of the spaceship
            reward : observed scalar reward
        """
        if spaceship_id not in self.data:
            raise ValueError(f"Unknown spaceship ID: {spaceship_id}, reward: {reward}, valid IDs are: {list(self.data.keys())}")
        self.data[spaceship_id].append(reward)

    def compute_posterior(self, spaceship_id):
        """
        Compute the posterior parameters for a given spaceship.
        """
        if self.model == 'perceptual':
            return 
            
        xi = np.array(self.data[spaceship_id])
        n = len(xi)

        if n == 0:
            return self.mu0, self.kappa0, self.alpha0, self.beta0

        x_bar = xi.mean()
        S = np.sum((xi - x_bar) ** 2)

        kappa_n = self.kappa0 + n
        mu_n = (self.kappa0 * self.mu0 + n * x_bar) / kappa_n
        alpha_n = self.alpha0 + n / 2
        beta_n = self.beta0 + 0.5 * S + (self.kappa0 * n * (x_bar - self.mu0) ** 2) / (2 * kappa_n)

        return mu_n, kappa_n, alpha_n, beta_n

    def posterior_predictive_mean(self, spaceship_id):
        """
        Return the expected value (mean) of the predictive distribution for a spaceship.
        """
        if self.model == 'perceptual':
            return 1.0
        mu_n, _, _, _ = self.compute_posterior(spaceship_id)
        return mu_n

    def sample_posterior(self, spaceship_id, n_samples=1000):
        """
        Sample possible mean rewards for a spaceship from the posterior predictive distribution.
        """
        mu_n, kappa_n, alpha_n, beta_n = self.compute_posterior(spaceship_id)

        tau_samples = gamma.rvs(a=alpha_n, scale=1 / beta_n, size=n_samples)
        mu_samples = norm.rvs(loc=mu_n, scale=1 / np.sqrt(kappa_n * tau_samples))
        #truncated normal distribution
        mu_samples = truncnorm.rvs(a=0, b=1, loc=mu_samples, scale=1 / np.sqrt(kappa_n * tau_samples))

        return mu_samples

    def reset(self, spaceship_id=None):
        """
        Reset observations for one or all spaceships.
        """
        if spaceship_id is None:
            for sid in self.data:
                self.data[sid] = []
        else:
            self.data[spaceship_id] = []

class NormativePolicy(NormativeValue):
    def __init__(self, spaceship_ids, mu0=.5, kappa0=1, alpha0=2, beta0=0.1,
                 burn_in=500, samples=1000, model=False, omega=0, beta=1, binary=True):
        NormativeValue.__init__(self, spaceship_ids, mu0, kappa0, alpha0, beta0)
        if model == 'gibbs':
            print('model is gibbs')
            self.perceptual = NormativePerceptual(burn_in=burn_in, samples=samples)
            self.perceptual.model = 'gibbs'
        else:
            # print('model is map')
            self.perceptual = SlopeMAPPerceptual(binary=binary)
            self.perceptual.model = 'map'
        self.perceptual_update = self.perceptual.perceptual_update
        self.predict_shield_p = self.perceptual.predict_shield_p
        
        self.omega = omega
        self.beta = beta
        self.model = model
        self.v_log_ratio_list = []
        self.p_log_ratio_list = []
        self.v_log_ratio = 0
        self.p_log_ratio = 0

    def expected_reward(self, spaceship_id, shield_strength):
        """
        Compute the expected reward for a spaceship given a shield strength z

        Parameters:
            spaceship_id : spaceship ID
            shield_strength : scalar z*

        Returns:
            Expected reward E[Y | z*, Z, B, X]
        """
        # Step 1: compute P(b=1 | z*) from probit model
        p_success = 1.0  # Default if no shield strength is provided
        if ~np.isnan(shield_strength) and self.model != 'value':
            p_success = self.perceptual.predict_shield_p(shield_strength)

        # Step 2: compute E[μ | X] from reward model
        E_mu = 1.0 
        if self.model != 'perceptual':
            mu_n, kappa_n, _, _ = self.compute_posterior(spaceship_id)
            E_mu = mu_n

        # Step 3: combine both
        return p_success, E_mu

    def get_spaceship_mean(self, spaceship_id):
        E_mu = 1.0
        if self.model != 'perceptual':
            mu_n, kappa_n, _, _ = self.compute_posterior(spaceship_id)
            E_mu = mu_n
        return E_mu
    
    def get_shield_probability(self, shield_strength):
        p_success = 1.0
        if ~np.isnan(shield_strength) and self.model != 'value':
            p_success = self.perceptual.predict_shield_p(shield_strength)
        return p_success

    def choose(self, spaceships, shields):
        """
        compute the expected reward for a spaceship given a shield strength z
        and select the one that tends to maximize it.
        """

        p1, p2 = [self.get_shield_probability(x) for x in shields]
        v1, v2 = [self.get_spaceship_mean(x) for x in spaceships]
    
        v_lr = np.log(v1) - np.log(v2)
        p_lr = np.log(p1) - np.log(p2)

        self.v_log_ratio_list.append(v_lr)  
        self.p_log_ratio_list.append(p_lr)

        # v_lr = v_lr / 0.59
        # p_lr = p_lr / 0.74        
        
        dv = (1-self.omega) * v_lr + self.omega * p_lr

        return int(np.random.random() > (1/(1 + np.exp(self.beta*-dv))))
    
    def get_likelihood(self, spaceships, shields):
        """
        computes the likelihood of both options
        given the shield strengths and  spaceships
        """

        p1, p2 = [self.get_shield_probability(x) for x in shields]
        v1, v2 = [self.get_spaceship_mean(x) for x in spaceships]
    
        v_lr = np.log(v1) - np.log(v2)
        p_lr = np.log(p1) - np.log(p2)

        # v_lr = v_lr / 0.59
        # p_lr = p_lr / 0.74        
        
        dv = (1-self.omega) * v_lr + self.omega * p_lr

        # estimated p of choosing option 1
        ep1 = 1/(1 + np.exp(self.beta*-dv))

        return [ep1, 1-ep1]
        
        
    def choose2(self, spaceships, shields):
        """
        compute the expected reward for a spaceship given a shield strength z
        and select the one that tends to maximize it.
        """

        p1, p2 = [self.get_shield_probability(x) for x in shields]
        v1, v2 = [self.get_spaceship_mean(x) for x in spaceships]
    
        self.v_log_ratio = np.log(v1) - np.log(v2)
        self.p_log_ratio = np.log(p1) - np.log(p2)

        # # v_lr = v_lr / np.std(self.v_log_ratio_list)
        # # p_lr = p_lr / np.std(self.p_log_ratio_list)

        # dv = (1-self.omega) * v_lr + self.omega * p_lr

        # return int(np.random.random() > (1/(1 + np.exp(self.beta*-dv))))
        
        # # not a dic now but two arrays
        expected_rewards = [
            sid*shield
            for sid, shield in zip([v1, v2], [p1, p2])
        ]
        return np.argmax(expected_rewards)

        #best_id = max(expected_rewards, key=expected_rewards.get)
        


In [68]:
import sys
sys.path.append('..')

from src.visualization import plot_settings
import pandas as pd
import numpy as np
import seaborn as sns

df = pd.read_csv('../data/processed/df.csv')

In [69]:
import itertools
from scipy.special import logsumexp
from scipy.stats import norm


def fit(x0, *args):
    # print('Running fit...')
    pid, fit_training, exp, model, session, ntrials, s, a, r, destroy, ff1, ff2, ff_values = args
    try:
        _ = len(x0)
    except:
        x0 = [x0]

    if model  in ('map', 'value', 'perceptual'):
        omega = x0[0]
        beta = x0[1]
        m = NormativePolicy(
            spaceship_ids=[280, 380, 500, 620, 720],
            omega=omega,
            beta=beta,
            binary=exp!='FullPilot14',
        )
    else:
        raise ValueError(f"Unknown model: {model}")
    
    logit = lambda x: 1 / (1 + np.exp(x))

    if model != 'random':

        ll = 0
        for t in range(ntrials):

            ff_chosen = [ff1[t], ff2[t]][a[t]]
            p_chosen = logit(-2*ff_chosen).round(2)

            if session[t] in (1, 3) and ('value' not in model):
                if exp == 'FullPilot14':
                    # spaceship_estimates = np.array([
                    #     m.posterior_predictive_mean(sid)
                    #     for sid in [280, 380, 500, 620, 720]
                    # ])
                    if session[t] == 1:
                        spaceship_estimate = .5
                    else:
                        spaceship_estimate = m.posterior_predictive_mean(s[t][a[t]])
                    m.perceptual_update(ff_chosen, r[t], spaceship_estimate=spaceship_estimate)
                else:
                    m.perceptual_update(ff_chosen, destroy[t])

            if session[t] in (0, 2, 3) and ('perceptual' not in model):
                if destroy[t]:
                    m.value_update(s[t][a[t]], r[t])
                if (not destroy[t]) and (exp == 'FullPilot13'):
                    m.value_update(s[t][a[t]], r[t]/2)

            if session[t] in (3, ) or fit_training:
                p_of_choices = m.get_likelihood(s[t], [ff1[t], ff2[t]])
                ll += np.log(p_of_choices[a[t]] + 1e-10)
    else:
        ll = 0
        for t in range(ntrials):
            ll += np.log(np.exp([.5, .5] - logsumexp([.5, .5])))[0]

    beta_prior_logpdf = norm.logpdf(beta, loc=1e5/2, scale=5)
    # check that beta prior is a valid number
    if np.isnan(beta_prior_logpdf):
        beta_prior_logpdf = 0
    elif np.isinf(beta_prior_logpdf):
        beta_prior_logpdf = 0
    # ll += beta_prior_logpdf

    return -ll,



In [71]:
from joblib import Parallel, delayed
import itertools
from pybads import BADS
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np
import ast

# Assuming df is predefined
exp = ['FullPilot12', 'FullPilot12_2', 'FullPilot13', 'FullPilot14']
agents = df[(df.expName.isin(exp))].agent.unique()
models = ['map']
parallel = True 
map_prolificID = {pid: idx for idx, pid in enumerate(df.prolificID.unique())}
slope = 2
logistic = lambda x: 1/(1+np.exp(-slope*x))
map_ff_values = {logistic(i).round(2):i for i in np.linspace(-1, 1, 9)}
map_ff_values[1] = 1
df['s'] = df.s.apply(ast.literal_eval)
# Function to optimize for one agent and model
def optimize_agent_model(agent, model, agent_data, fit_training=False):
    # print('Running optimization for', agent, model)
    df2 = agent_data
    # drop rows where session==3 and pair is not in [0,1]
    
    # Condition to drop rows
    if not fit_training:
        condition = ~((df2['session'] == 3) & (~df2['pair'].isin([0, 1])))
    else:
        condition = (df2['session'].isin([0, 2]))
    # # # Drop rows meeting the condition, keeping other rows intact
    df2 = df2.loc[condition]
    # df2 = df2[df2.session==3]

    s, a, r = df2.s.values, df2.a.values, df2.r.values
    ff1, ff2 = df2.ff1.values, df2.ff2.values
    session = df2.session.values
    destroy = df2.destroy.values
    ntrials = df2.shape[0]
    exp = df2.expName.values[0]
    pid = map_prolificID[df2.prolificID.values[0]]

    def target(x):
        return fit(x, *(pid, fit_training, exp, model, session, ntrials, s, a, r, destroy, ff1, ff2, map_ff_values.values()))[0]

    # if any(s in model for s in ['single', 'value', 'perceptual', 'arbitration']):
    #     lower_bounds = np.array([-1e3])
    #     upper_bounds = np.array([1e3])
    #     x0 = np.array([1])

    if  any(s in model for s in ['absolute', 'log-ratio', 'absolute-var', 'map', 'perceptual', 'value']):
        lower_bounds = np.array([0, 0])
        upper_bounds = np.array([1, 1e4])
        x0 = np.array([0.5, 5])
   
    # To prevent triggering the "bads:StrictBoundsTooClose" or similar errors:
    # Ensure plausible bounds are within the lower and upper bounds.

    plausible_lower_bounds = np.array([0.05, 1])  # Should be slightly above lower_bounds but still below upper_bounds
    plausible_upper_bounds = np.array([0.95, 50])  # Should be slightly below upper_bounds but still above plausible_lower_bounds

    bads = BADS(
        target,
        x0=x0,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
        plausible_lower_bounds=plausible_lower_bounds,
        plausible_upper_bounds=plausible_upper_bounds,
        options={'display': 'off'}
    )

    optimize_result = bads.optimize()
    return {'agent': agent, 'model': model, 'omega': optimize_result.x[0], 
            'beta': optimize_result.x[1], 'll': optimize_result.fval, 
             'ntrials': ntrials, 'exp': exp, 'prolificID': df2.prolificID.values[0]}


from joblib import Parallel, delayed
from tqdm_joblib import tqdm_joblib
from tqdm_joblib import ParallelPbar
import itertools

# Prepare list of all agent-model pairs
tasks = itertools.product(agents, models)


results = []
parallel = True
fit_training = False

if parallel:
    # Parallel execution with notebook-friendly progress bar
    results = []
    for result in ParallelPbar('N sub')(n_jobs=-2, # Use all available cores except 1 to keep the system responsive
                            backend='loky' 
    )(delayed(optimize_agent_model)(agent, model, df[df.agent == agent].sort_values('t'), fit_training) for agent, model in tasks):
        results.append(result)

else:

    # non parallel version
    results = []
    for agent, model in tasks:
        results.append(optimize_agent_model(agent, model, df[df.agent == agent].sort_values('t')))
        # pbar.update()

df_fit = pd.DataFrame(results)



N sub:   0%|          | 0/245 [00:00<?, ?it/s]

In [ ]:
import scipy.stats as stats

session = 3
df_ = df[(df.session==session)].groupby(['prolificID'], as_index=False).mean(numeric_only=True)

df_['opti_ff'] = df_.opti_ff.astype(float)
df_['opti_ss'] = df_.opti_ss.astype(float)
df['opti_ff'] = df['opti_ff'].astype(float)
df['opti_ss'] = df['opti_ss'].astype(float)
         
df2 = df[(df.session==session)]


def get_group(row):
    opti_ff = df2[df2.prolificID==row.prolificID].opti_ff
    opti_ss = df2[df2.prolificID==row.prolificID].opti_ss
    p_ss = stats.ttest_1samp(opti_ss, 0.5, alternative='greater').pvalue < 0.05 
    p_ff = stats.ttest_1samp(opti_ff, 0.5, alternative='greater').pvalue < 0.05

    if p_ss and p_ff:
        return 'combined'
        
    
    if p_ff:
        return 'perceptual'
    if p_ss:
        return 'value'

    return 'random'


df_['group'] = df_.apply(get_group, axis=1)
df['group'] = df['prolificID'].map(df_.set_index('prolificID')['group'])

print(len(df.prolificID.unique()))

245


In [75]:
# in df_fit remove random subjects
df_random = df[df.group=='random'].prolificID.unique()
df_fit = df_fit[~df_fit.prolificID.isin(df_random)]

# equal thirds split based on omega
df_fit['group'] = df_fit.apply(
    lambda agent: 'perceptual' if agent.omega > .66 else 'value'
      if agent.omega < .33 else 'combined', axis=1)

C:\Users\basil\AppData\Local\Temp\ipykernel_33240\1378077520.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fit['group'] = df_fit.apply(


In [76]:
# show distribution of omega values per experiment
df_fit.groupby(['exp', 'group']).group.value_counts()

exp            group     
FullPilot12    combined      13
               perceptual    31
               value          7
FullPilot12_2  combined      13
               perceptual    30
               value          7
FullPilot13    combined       5
               perceptual    22
               value         22
FullPilot14    combined      20
               perceptual    29
               value          5
Name: count, dtype: int64

In [77]:
df_fit.to_csv('../data/processed/omega_fit_full.csv', index=False)